# 02 - Battery Sizing

Battery mass is one of the largest weight fractions in an eVTOL aircraft — often 25–35% of MTOW. This notebook explores how `evtolpy` models battery energy density and computes the required battery mass.

The fundamental sizing equation is:

$$m_{battery} = \frac{E_{total}}{e_{usable,EOL}}$$

where $E_{total}$ is the total mission energy (primary + reserve) and $e_{usable,EOL}$ is the end-of-life usable specific energy at the pack level. Because battery mass affects MTOW, which affects energy, this creates a **circular dependency** resolved by the MTOW iteration loop.

In [ ]:
import sys
sys.path.append('../../evtol')
from aircraft import Aircraft

aircraft = Aircraft('../../analysis/cfg-case-study/low-altitude-1500-ft/archer-midnight/30-miles/Archer-Midnight-1500-30.json')

## Battery Parameters

The `Power` class models battery energy density with several **derating factors** that reduce cell-level performance to pack-level usable energy:

| Parameter | Symbol | Description |
|---|---|---|
| Specific energy | $e_{cell}$ | Cell-level gravimetric energy density (Wh/kg) |
| Integration factor | $K_{int}$ | Pack overhead — thermal management, casing, BMS (typically 0.7–0.85) |
| Inaccessible energy | $f_{inacc}$ | Energy lost to internal resistance at high discharge rates |
| End-of-life capacity | $f_{EOL}$ | Capacity retention after calendar + cycle aging (typically 0.80) |

Each factor compounds multiplicatively, so the effective pack-level energy can be 40–50% below the cell specification.

In [ ]:
pwr = aircraft.power

print(f"Cell Specific Energy:     {pwr.batt_spec_energy_w_h_p_kg:.1f} Wh/kg")
print(f"Inaccessible Energy:      {pwr.batt_inaccessible_energy_frac:.0%}")
print(f"End-of-Life Capacity:     {pwr.batt_eol_capacity:.0%}")
print(f"Integration Factor:       {pwr.batt_int_factor:.2f}")
print(f"EPU Efficiency:           {pwr.epu_effic:.2f}")
print(f"Hover Power Efficiency:   {pwr.hover_power_effic:.2f}")

## Usable Specific Energy

The usable specific energy at the pack level accounts for all derating factors in sequence:

$$e_{usable,BOL} = K_{int} \times (1 - f_{inacc}) \times e_{cell}$$
$$e_{usable,EOL} = f_{EOL} \times e_{usable,BOL}$$

This is the energy density that actually determines battery mass. The "derating waterfall" below shows how each factor reduces the effective energy density from cell spec to pack EOL usable.

In [ ]:
bol = pwr.batt_int_factor * (1 - pwr.batt_inaccessible_energy_frac) * pwr.batt_spec_energy_w_h_p_kg
eol = pwr.batt_eol_capacity * bol

print(f"Cell-level specific energy:     {pwr.batt_spec_energy_w_h_p_kg:.1f} Wh/kg")
print(f"Pack-level BOL usable energy:   {bol:.1f} Wh/kg")
print(f"Pack-level EOL usable energy:   {eol:.1f} Wh/kg")
print(f"")
print(f"Overall derating ratio: {eol/pwr.batt_spec_energy_w_h_p_kg:.1%} of cell-level")

# Derating waterfall visualization
import matplotlib.pyplot as plt

stages = ['Cell Spec', 'After\nIntegration', 'After\nInaccessible', 'After\nEOL']
values = [
    pwr.batt_spec_energy_w_h_p_kg,
    pwr.batt_int_factor * pwr.batt_spec_energy_w_h_p_kg,
    pwr.batt_int_factor * (1 - pwr.batt_inaccessible_energy_frac) * pwr.batt_spec_energy_w_h_p_kg,
    eol
]
losses = [0] + [values[i-1] - values[i] for i in range(1, len(values))]

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#4e79a7'] + ['#e15759'] * 3
remaining_bars = ax.bar(stages, values, color=bar_colors, edgecolor='white', alpha=0.85)

# Add loss annotations
for i in range(1, len(stages)):
    ax.annotate(f'-{losses[i]:.1f}\nWh/kg',
                xy=(i, values[i] + (values[i-1] - values[i])/2),
                fontsize=9, ha='center', va='center', color='white', fontweight='bold')

for bar, val in zip(remaining_bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Specific Energy (Wh/kg)')
ax.set_title('Battery Energy Derating Waterfall')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Battery Mass Calculation

The battery mass is sized to carry enough energy for the entire mission (primary + reserve):

$$m_{battery} = \frac{(E_{primary} + E_{reserve}) \times 1000}{e_{usable,EOL}}$$

where the factor of 1000 converts kWh to Wh. The reserve mission adds a substantial energy requirement — it represents a regulatory diversion scenario where the aircraft must fly to an alternate landing site.

In [ ]:
total_primary = aircraft.total_mission_energy_kw_hr
total_reserve = aircraft.total_reserve_mission_energy_kw_hr
total_energy = total_primary + total_reserve

print(f"Primary mission energy:  {total_primary:.2f} kWh")
print(f"Reserve mission energy:  {total_reserve:.2f} kWh")
print(f"Total energy required:   {total_energy:.2f} kWh")
print(f"")
print(f"Battery mass:            {aircraft.battery_mass_kg:.2f} kg")
print(f"  ({100*aircraft.battery_mass_kg/aircraft.max_takeoff_mass_kg:.1f}% of MTOW)")

## Impact of Battery Technology

Battery specific energy is a critical driver of eVTOL feasibility. The plot below shows how battery mass varies with cell-level specific energy (holding total energy constant for illustration).

The relationship is **hyperbolic** ($m \propto 1/e$), meaning:
- Improvements from 150 → 200 Wh/kg have a much larger effect than 350 → 400 Wh/kg
- There are diminishing returns as technology improves — at high specific energy, battery mass becomes a small fraction of MTOW and other masses dominate

In [ ]:
import matplotlib.pyplot as plt

spec_energies = range(150, 450, 25)  # Wh/kg
batt_masses = []
batt_fracs = []
for se in spec_energies:
    eol_usable = pwr.batt_eol_capacity * pwr.batt_int_factor * (1 - pwr.batt_inaccessible_energy_frac) * se
    bm = total_energy * 1000 / eol_usable  # energy in Wh / Wh/kg = kg
    batt_masses.append(bm)
    batt_fracs.append(100 * bm / aircraft.max_takeoff_mass_kg)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(list(spec_energies), batt_masses, 'o-', color='steelblue')
ax1.axvline(x=pwr.batt_spec_energy_w_h_p_kg, color='coral', linestyle='--',
           label=f'Current: {pwr.batt_spec_energy_w_h_p_kg:.0f} Wh/kg')
ax1.axhline(y=aircraft.battery_mass_kg, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Cell Specific Energy (Wh/kg)')
ax1.set_ylabel('Battery Mass (kg)')
ax1.set_title('Battery Mass vs. Cell Specific Energy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(list(spec_energies), batt_fracs, 's-', color='seagreen')
ax2.axvline(x=pwr.batt_spec_energy_w_h_p_kg, color='coral', linestyle='--',
           label=f'Current: {pwr.batt_spec_energy_w_h_p_kg:.0f} Wh/kg')
ax2.axhline(y=35, color='gray', linestyle=':', alpha=0.5, label='35% threshold')
ax2.set_xlabel('Cell Specific Energy (Wh/kg)')
ax2.set_ylabel('Battery Mass Fraction (%)')
ax2.set_title('Battery Fraction vs. Cell Specific Energy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Archer Midnight — {total_energy:.1f} kWh total energy', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Summary

Key battery sizing insights:

- **Derating waterfall**: Cell-level specific energy is reduced by integration factor, inaccessible energy, and EOL capacity — the pack-level usable energy is typically 50–60% of the cell spec
- **Battery mass** = total mission energy ÷ usable specific energy — this is the single largest variable mass component
- **Battery fraction** above ~35% of MTOW approaches practical limits — the mass snowball effect makes further range extension increasingly expensive
- **Technology sensitivity** is hyperbolic: early improvements in specific energy have dramatic effects; above ~350 Wh/kg, diminishing returns set in
- Battery mass feeds back into MTOW, which increases structural mass, which increases power, which increases energy — the **snowball effect**

**Next:** Move on to [08 - Autonomous Battery Units](../08%20-%20Autonomous%20Battery%20Units/) to explore the ABU concept.